# 07 — Process trade data

Canonicalise monthly JODI product trade and aggregate to annual thousand tonnes. If the raw download is absent, use the explicitly provisional seed export series only for notebook development.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.jodi import read_secondary_zip, canonicalise_secondary, filter_portugal_fuels, annualise
from portugal_refining_resilience.validation import assert_nonnegative, assert_unique


In [ ]:
raw_zip = PATHS.raw / "jodi" / "world_secondary_csv.zip"
if raw_zip.exists():
    raw = read_secondary_zip(raw_zip)
    canonical = canonicalise_secondary(raw)
    selected = filter_portugal_fuels(canonical, flows=("exports", "imports"))
    annual = annualise(selected)
    # Standardise flow labels after keeping original labels available upstream.
    annual["flow"] = np.where(annual["flow_canonical"].str.contains("export"), "exports", "imports")
    annual = annual.rename(columns={"product_canonical": "product"})
    annual = annual[["year", "country", "product", "flow", "value_kt", "source"]]
    annual["status"] = "downloaded"
else:
    print("Raw JODI ZIP not found: using the provisional seed exports only.")
    annual = pd.read_csv(PATHS.processed / "jodi_portugal_fuel_exports_2005_2024_seed.csv")

annual = annual.loc[annual["year"].between(2005, 2024)].copy()
assert_nonnegative(annual, ["value_kt"])
assert_unique(annual, ["year", "product", "flow"])
display(annual.tail(10))
persist_dataframe(annual, PATHS.processed / "fuel_trade_annual.csv", key_columns=["year", "product", "flow"], metadata={"unit": "kt"})
